# 9장. 회귀 분석으로 숫자 예측하기

이 Notebook의 핵심은 높은 점수를 만드는 것이 아니라 다음 순서를 지키는 것입니다.

`예측 시점 → 누수 방지 → 시간 분할 → Train TimeSeriesSplit → 모델 선택 고정 → Final Test → Baseline 비교`

**Final Test 결과를 보고 모델을 고르지 않습니다.**

## 0. 실행 전 확인

Chapter09는 Chapter05의 오류 탐지용 전용 Raw가 아니라 **공통 프로젝트 `data/raw`**를 사용합니다. 먼저 프로젝트 루트에서 다음 명령을 실행해 관계 검증을 통과한 모델링 입력을 준비합니다.

```powershell
python scripts/prepare_ch09_data.py
```

이 명령은 공통 `data/raw`를 전처리하고 FK 관계를 검증한 뒤 `data/processed/*_clean.csv`를 생성합니다.

In [1]:
from pathlib import Path
import sys
import pandas as pd

CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = Path(r"C:\dev\llm-data-analysis-course-ch09")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
REPORT_DIR = PROJECT_ROOT / 'reports'

from src.regression import (
    FEATURE_COLUMNS,
    FORBIDDEN_FEATURES,
    TARGET_COLUMN,
    build_feature_audit,
    build_leakage_checklist,
    build_regression_dataset,
    build_regression_validation,
    build_split_summary,
    create_prediction_result,
    cross_validate_regression_models,
    load_regression_source_data,
    make_regression_models,
    run_regression_analysis,
    save_regression_outputs,
    select_diagnostic_model,
    split_model_data_by_time,
    train_and_evaluate_models,
    validate_feature_columns,
)

print('PROJECT_ROOT:', PROJECT_ROOT)
print('PROCESSED_DIR:', PROCESSED_DIR)

PROJECT_ROOT: C:\dev\llm-data-analysis-course-ch09
PROCESSED_DIR: C:\dev\llm-data-analysis-course-ch09\data\processed


## 1. 예측 문제와 예측 시점

- Target: 주문 1건의 주문 상세 금액 합계 `order_total`
- 예측 시점: 주문 메타데이터와 고객의 비식별 특성은 알 수 있지만 주문 상세 수량·단가·금액은 모델에 제공하지 않는 시점
- 목표 계산 재료와 사후 정보, 식별자는 feature에서 제외합니다.

- **예측 대상** — 주문 1건의 상품 금액 합계(`order_total`). 각 주문 상세의 수량과 단가를 곱한 금액을 주문별로 합산한다.
- **예측 단위** — 주문 1건
- **예측 시점** — 결제수단이 선택되어 주문 기본 정보를 확인할 수 있지만, 주문 상세와 총액은 모델에 제공되지 않은 시점으로 가정한다.
- **예측 시점에 사용할 수 있는 정보** — 주문 월과 요일, 고객의 나이·성별·도시, 선택한 결제수단
- **예측 시점에 사용할 수 없는 정보** — 주문 상세의 수량·단가·금액, 주문 총액, 주문 이후 확정되는 최종 주문 상태

In [2]:
print('Target:', TARGET_COLUMN)
print('Allowed features:', FEATURE_COLUMNS)
print('Forbidden features:', sorted(FORBIDDEN_FEATURES))
validate_feature_columns()
print('Feature leakage gate: PASS')

Target: order_total
Allowed features: ['order_month', 'order_dayofweek', 'age', 'payment_method', 'gender', 'city']
Forbidden features: ['avg_unit_price', 'customer_id', 'item_count', 'line_total', 'order_id', 'order_status', 'order_total', 'product_id', 'quantity', 'total_quantity', 'unit_price']
Feature leakage gate: PASS


## 2. 검증된 전처리 데이터를 불러옵니다

In [3]:
data = load_regression_source_data(PROCESSED_DIR)
for name, frame in data.items():
    print(name, frame.shape)

customers (150, 6)
orders (300, 7)
order_items (764, 6)


## 3. 주문 단위 Target을 만들고 관계를 검증합니다

`line_total = quantity × unit_price`를 확인하고, 주문과 Target은 `one_to_one`, 주문과 고객은 `many_to_one` 관계가 깨지지 않는지 확인합니다.

In [4]:
model_data = build_regression_dataset(
    customers=data['customers'],
    orders=data['orders'],
    order_items=data['order_items'],
)
display(model_data.head())
print('Model rows:', len(model_data))

,order_id,customer_id,order_date,payment_method,order_status,order_month,order_dayofweek,order_total,gender,age,city
0,238,124,2025-07-09,bank_transfer,completed,7,2,361000,M,67,대구
1,79,62,2025-07-11,bank_transfer,refunded,7,4,283000,F,63,광주
2,12,98,2025-07-12,kakao_pay,completed,7,5,444000,F,61,인천
3,169,9,2025-07-12,kakao_pay,completed,7,5,448000,M,69,서울
4,152,62,2025-07-13,naver_pay,refunded,7,6,985000,F,63,광주


Model rows: 300


## 4. Feature Audit을 확인합니다

In [5]:
feature_audit = build_feature_audit()
display(feature_audit)

,column,selected,role,reason
0,order_month,True,allowed_feature,교육용 예측 시점에 사용 가능하다고 가정
1,order_dayofweek,True,allowed_feature,교육용 예측 시점에 사용 가능하다고 가정
2,age,True,allowed_feature,교육용 예측 시점에 사용 가능하다고 가정
3,payment_method,True,allowed_feature,교육용 예측 시점에 사용 가능하다고 가정
4,gender,True,allowed_feature,교육용 예측 시점에 사용 가능하다고 가정
5,city,True,allowed_feature,교육용 예측 시점에 사용 가능하다고 가정
6,avg_unit_price,False,forbidden,주문 상세에서 만든 목표 대리 변수
7,customer_id,False,forbidden,고객 식별자
8,item_count,False,forbidden,주문 상세가 확인된 뒤 계산되는 사후 집계
9,line_total,False,forbidden,목표값을 구성하는 주문 상세 금액


## 5. 시간 순서로 Train / Final Test를 분리합니다

같은 달력 날짜를 양쪽에 나누지 않습니다. Final Test는 모델 선택이 끝날 때까지 사용하지 않습니다.

In [6]:
train_data, test_data = split_model_data_by_time(model_data, test_size=0.2)
split_summary = build_split_summary(train_data, test_data)
display(split_summary)
assert train_data['order_date'].max().normalize() < test_data['order_date'].min().normalize()
print('Chronological split gate: PASS')

,split,rows,ratio_pct,start_date,end_date
0,train,244,81.33,2025-07-09,2026-04-13
1,test,56,18.67,2026-04-14,2026-07-08


Chronological split gate: PASS


In [7]:
X_train = train_data[FEATURE_COLUMNS].copy()
X_test = test_data[FEATURE_COLUMNS].copy()
y_train = train_data[TARGET_COLUMN].copy()
y_test = test_data[TARGET_COLUMN].copy()
print('Train:', X_train.shape, 'Final Test:', X_test.shape)

Train: (244, 6) Final Test: (56, 6)


## 6. Pipeline 후보 모델을 준비합니다

숫자형 결측 대체·표준화와 범주형 결측 대체·One-Hot Encoding은 모델 Pipeline 안에서 각 Train Fold로만 학습됩니다.

In [8]:
models = make_regression_models(random_state=42)
print(list(models))

['Baseline Mean', 'Linear Regression', 'Random Forest']


## 7. Train 기간의 TimeSeriesSplit으로 후보를 비교합니다

여기까지는 Final Test의 `y_test`를 모델 선택에 사용하지 않습니다.

In [9]:
cv_summary = cross_validate_regression_models(
    models=models,
    X_train=X_train,
    y_train=y_train,
)
display(cv_summary)

,model,n_splits,cv_MAE_mean,cv_MAE_std,cv_R2_mean
0,Baseline Mean,5,423004.808743,44153.043466,-0.022821
1,Random Forest,5,437352.803929,26318.793429,-0.109687
2,Linear Regression,5,461386.286905,77792.799324,-0.310517


## 8. 비베이스라인 모델 선택을 고정합니다

Linear Regression과 Random Forest 중 Train CV의 평균 MAE가 더 낮은 모델을 선택합니다. **이 시점까지 Final Test 성능은 보지 않습니다.**

In [10]:
selected_model_name = select_diagnostic_model(cv_summary)
print('Frozen selected model:', selected_model_name)

Frozen selected model: Random Forest


## 9. 이제 Final Test에서 Baseline과 고정 모델만 평가합니다

Final Test 결과가 기대와 다르더라도 같은 Test를 보고 다른 후보로 교체하지 않습니다.

In [11]:
model_comparison, predictions = train_and_evaluate_models(
    models=models,
    selected_model_name=selected_model_name,
    X_train=X_train,
    X_test=X_test,
    y_train=y_train,
    y_test=y_test,
)
display(model_comparison)

,model,selection_role,train_MAE,test_MAE,test_RMSE,test_R2,MAE_improvement_vs_baseline_pct
0,Baseline Mean,baseline,416266.191884,443449.648712,522189.699620,-0.001121,0.00
1,Random Forest,selected_by_train_cv,326438.478132,458819.917234,548190.446723,-0.103298,-3.47


### 지표 읽기

- MAE: 평균적으로 얼마만큼 틀리는가
- RMSE: 큰 오차에 더 민감
- R²: 평균 예측 대비 설명력, 음수가 될 수 있음
- MAE 개선율: Dummy baseline 대비 고정 모델의 Final Test MAE 개선 정도

## 10. 실제값·예측값·잔차를 진단합니다

In [ ]:
prediction_result = create_prediction_result(
    test_data=test_data,
    y_test=y_test,
    y_pred=predictions[selected_model_name],
    model_name=selected_model_name,
)
display(prediction_result.head(10))
print('주의: order_id가 포함된 이 표는 내부 진단용입니다.')

## 11. 자동 Validation Evidence를 만듭니다

In [13]:
validation = build_regression_validation(
    train_data=train_data,
    test_data=test_data,
    cv_summary=cv_summary,
    selected_model_name=selected_model_name,
    model_comparison=model_comparison,
)
display(validation)
assert validation['status'].eq('PASS').all()

,check,value,status
0,forbidden_feature_overlap,0,PASS
1,strict_train_before_test,True,PASS
2,selected_model_exists_in_train_cv,True,PASS
3,final_test_contains_baseline,True,PASS
4,final_test_contains_frozen_selected_model,True,PASS
5,test_rows_for_r2,56,PASS


## 12. Evidence·내부 진단·공개 보고서를 구분해 저장합니다

In [14]:
checklist = build_leakage_checklist()
output_paths = save_regression_outputs(
    model_data=model_data,
    split_summary=split_summary,
    feature_audit=feature_audit,
    cv_summary=cv_summary,
    selected_model_name=selected_model_name,
    model_comparison=model_comparison,
    prediction_result=prediction_result,
    validation=validation,
    checklist=checklist,
    report_dir=REPORT_DIR,
)
for name, path in output_paths.items():
    print(name, '->', path)

model_data_internal -> C:\dev\llm-data-analysis-course-ch09\reports\ch09_regression_model_data_internal.csv
split_summary -> C:\dev\llm-data-analysis-course-ch09\reports\ch09_regression_split_summary.csv
feature_audit -> C:\dev\llm-data-analysis-course-ch09\reports\ch09_regression_feature_audit.csv
cv_summary -> C:\dev\llm-data-analysis-course-ch09\reports\ch09_regression_cv_summary.csv
model_comparison -> C:\dev\llm-data-analysis-course-ch09\reports\ch09_regression_model_comparison.csv
predictions_internal -> C:\dev\llm-data-analysis-course-ch09\reports\ch09_regression_predictions_internal.csv
validation -> C:\dev\llm-data-analysis-course-ch09\reports\ch09_regression_validation.csv
checklist -> C:\dev\llm-data-analysis-course-ch09\reports\ch09_regression_checklist.csv
report -> C:\dev\llm-data-analysis-course-ch09\reports\ch09_regression_report.md
actual_vs_predicted_figure -> C:\dev\llm-data-analysis-course-ch09\reports\figures\ch09_actual_vs_predicted.png
residual_figure -> C:\dev\l

## 13. 전체 공통 함수를 새 실행으로 다시 확인합니다

셀의 메모리 상태가 아니라 공통 함수가 같은 순서를 재현하는지 확인합니다.

In [15]:
result = run_regression_analysis(
    processed_dir=PROCESSED_DIR,
    report_dir=REPORT_DIR,
    test_size=0.2,
    random_state=42,
)
print('Selected by Train CV:', result['selected_model_name'])
display(result['validation'])

Selected by Train CV: Random Forest


,check,value,status
0,forbidden_feature_overlap,0,PASS
1,strict_train_before_test,True,PASS
2,selected_model_exists_in_train_cv,True,PASS
3,final_test_contains_baseline,True,PASS
4,final_test_contains_frozen_selected_model,True,PASS
5,test_rows_for_r2,56,PASS


## 14. 해석

낮은 성능이나 음수 R²도 숨기지 않습니다. 현재 예측 시점에 허용한 정보가 약하다면 `현재 데이터로는 모델 사용 보류`도 올바른 결론입니다.

LLM이 만든 코드도 같은 예측 시점·누수·Pipeline·Train CV·Final Test 보호 기준으로 검토합니다.

## 15. 완료 체크

- [ ] `python scripts/prepare_ch09_data.py`로 공통 모델링 입력을 준비했다.
- [ ] Target과 예측 시점을 설명할 수 있다.
- [ ] 금지 feature가 입력에 없음을 확인했다.
- [ ] 같은 날짜가 Train과 Final Test에 동시에 들어가지 않는다.
- [ ] 전처리는 Pipeline 내부에서 학습된다.
- [ ] Train TimeSeriesSplit으로 후보를 비교했다.
- [ ] Final Test 전에 Selected Model을 고정했다.
- [ ] Final Test에서는 Baseline과 Frozen Model만 비교했다.
- [ ] MAE, RMSE, R²를 함께 읽었다.
- [ ] 내부 식별자 결과와 공개 보고서를 구분했다.
- [ ] Validation Evidence가 모두 PASS다.

# Chapter 09 답안. 회귀 분석으로 숫자 예측하기

## 제출 정보

- 이름: 오현우
- GitHub ID: dhgusdn0312
- 작성일: 2026.11.10
- 최종 Notebook URL: https://github.com/dhgusdn0312/llm-data-analysis-study/blob/main/chapter09/chapter09.ipynb

## 1. 예측 문제 정의

- Target: 주문별 상품 금액 합계(`order_total`)
- 예측 단위: 주문 1건
- 예측 시점: 결제수단이 선택되어 주문 기본 정보를 확인할 수 있지만, 주문 상세와 총액은 모델에 제공되지 않은 시점으로 가정했다.
- 예측 시점에 알 수 있는 정보: 주문 월과 요일, 고객의 나이·성별·도시, 결제수단
- 예측 시점에 알 수 없는 정보: 모델에 제공되지 않은 상품별 수량·단가·금액과 주문 총액, 주문 이후 확정되는 최종 주문 상태

### 나의 해석과 판단

주문 기본 정보와 고객 특성만으로 주문 금액을 어느 정도 예측할 수 있는지 확인하기 위해 이 시점을 정했다.

## 2. Feature Leakage Audit

| feature | 예측 시점 사용 가능 | Target 계산 재료 | 사후 정보/ID | 최종 사용 | 판단 이유 |
| --- | --- | --- | --- | --- | --- |
| `order_month`, `order_dayofweek` | 가능 | 아니오 | 해당 없음 | 사용 | 예측 시점의 주문 날짜에서 알 수 있다. |
| `age`, `gender`, `city` | 가능으로 가정 | 아니오 | 해당 없음 | 사용 | 실습에서 예측 시점에 확인 가능한 고객 특성으로 가정했다. |
| `payment_method` | 가능으로 가정 | 아니오 | 해당 없음 | 사용 | 결제수단이 선택된 시점으로 정했다. |
| `order_total` | 불가 | Target 자체 | 해당 없음 | 제외 | 예측할 값이 그대로 들어간다. |
| `line_total`, `quantity`, `unit_price` | 불가 | 예 | 주문 상세 | 제외 | 주문 총액을 직접 계산할 수 있다. |
| `avg_unit_price`, `total_quantity` | 불가 | 간접 관련 | 주문 상세 집계 | 제외 | 주문 상세에서 계산한 값으로 목표 금액의 단서가 된다. |
| `item_count` | 불가 | 아니오 | 주문 상세 집계 | 제외 | 주문 상세가 확인된 뒤 계산할 수 있다. |
| `order_status` | 불가 | 아니오 | 사후 정보 | 제외 | 예측 이후 확정될 수 있다. |
| `order_id`, `customer_id` | 식별에만 사용 | 아니오 | ID | 제외 | 주문과 고객을 연결하는 식별자다. |
| `product_id` | 불가 | 아니오 | 주문 상세 ID | 제외 | 주문 상세가 확인되어야 알 수 있는 상품 식별자다. |

- 금지 feature overlap 개수: 0개
- 가장 위험하다고 판단한 feature: `order_total`
- 이유: Target 자체이므로 입력에 들어가면 정답이 그대로 노출된다.

![Leakage 점검](images/step02_leakage.png)

## 3. Target 생성과 관계 검증

```text
line_total = quantity × unit_price
order_total = 같은 order_id의 line_total 합계
```

- `line_total` 검증 결과: 공식 검증 셀이 오류 없이 실행됐다.
- `orders` ↔ `order_total` 관계 검증 결과: 주문별 합계가 주문 1건에 하나씩 연결되는 `one_to_one` 관계를 통과했다.
- `orders` → `customers` 관계 검증 결과: 여러 주문이 한 고객에 연결되는 `many_to_one` 관계를 통과했다.
- 병합 후 분석표: 주문 300건이 300행으로 유지됐다.

### 나의 해석과 판단

병합 키가 중복되면 같은 주문이 여러 행으로 늘어날 수 있고, 연결할 키가 없으면 주문이 빠질 수 있다. 병합 관계와 병합 후 행 수를 함께 확인했다.

## 4. Train / Final Test 분할

| 구분 | 시작일 | 종료일 | 행 수 |
| --- | --- | --- | ---: |
| Train | 2025-07-09 | 2026-04-13 | 244 |
| Final Test | 2026-04-14 | 2026-07-08 | 56 |

- 같은 달력 날짜 중복 여부: 없음
- 시간순 분할 검증: PASS

### 분할 판단

과거 주문으로 학습한 모델이 이후 주문에서도 얼마나 맞는지 평가하기 위해 날짜순으로 분리했다.

![시간 분할](images/step04_split.png)

## 5. Train TimeSeriesSplit 후보 비교

| 모델 | CV MAE 평균(원) | CV MAE 표준편차(원) | CV R² 평균 | 비고 |
| --- | ---: | ---: | ---: | --- |
| Baseline Mean | 423,005 | 44,153 | -0.0228 | 평균 MAE가 가장 낮음 |
| Linear Regression | 461,386 | 77,793 | -0.3105 | 비베이스라인 후보 |
| Random Forest | 437,353 | 26,319 | -0.1097 | 비베이스라인 중 평균 MAE가 가장 낮음 |

### Train CV로 선택한 비베이스라인 모델

- Selected Model: Random Forest
- 선택 기준: Train 기간의 5개 시계열 검증 구간에서 Linear Regression보다 평균 MAE가 낮았다.
- Final Test 결과를 보기 전에 선택했는가: 예

### 나의 해석과 판단

MAE 평균은 검증 구간 전반의 오차 수준을 보여주고, 표준편차는 구간마다 오차가 얼마나 달라졌는지 보여준다. Random Forest의 구간별 변동은 세 모델 중 가장 작았지만, 평균 MAE는 Baseline보다 높았다.

## 6. Final Test: Baseline vs Frozen Model

| 모델 | selection role | Train MAE(원) | Test MAE(원) | Test RMSE(원) | Test R² | Baseline 대비 MAE 개선율 |
| --- | --- | ---: | ---: | ---: | ---: | ---: |
| Baseline Mean | baseline | 416,266 | 443,450 | 522,190 | -0.0011 | 0.00% |
| Random Forest | selected_by_train_cv | 326,438 | 458,820 | 548,190 | -0.1033 | -3.47% |

![최종 회귀 평가](images/step06_final_metrics.png)

### 결과 관찰

Final Test에서 Random Forest는 주문당 평균 약 458,820원 차이가 났다. Baseline보다 Test MAE가 약 15,370원 높았다.

### 나의 해석과 판단

- Random Forest의 Test MAE는 Baseline보다 3.47% 높아 개선 효과가 없었다.
- Random Forest의 RMSE는 MAE보다 약 89,370원 높았다. 실제 오차 표에서도 100만 원 이상 빗나간 사례가 확인됐다.
- Random Forest의 R²는 -0.1033이었다. 제곱오차 기준으로 Final Test 실제값의 평균을 일괄 예측했을 때보다 오차가 컸다는 뜻이다.

### 업무·분석적 의미

주문 한 건의 금액을 기준으로 판단해야 하는 업무에서는 약 46만 원의 평균 오차가 부담이 될 수 있다. 현재 결과로는 Random Forest를 도입할 근거가 부족하다.

## 7. 대표 오차 사례

### 사례 1

- 실제값: 1,877,000원
- 예측값: 605,529원
- residual: +1,271,471원
- absolute error: 1,271,471원
- 관찰: 고액 주문의 금액을 크게 낮게 예측했다.
- 가능한 이유 후보: 사용한 입력 변수만으로 고액 주문을 구분하기 어려웠을 수 있다.
- 추가 확인: 주문 상세의 수량·단가·상품 구성을 진단용으로 확인하고, 실제 금액대별 오차를 비교한다.

### 사례 2

- 실제값: 10,000원
- 예측값: 897,415원
- residual: -887,415원
- absolute error: 887,415원
- 관찰: 소액 주문의 금액을 크게 높게 예측했다.
- 가능한 이유 후보: 학습 데이터에서 이 주문과 비슷한 입력 조건을 가진 소액 주문이 적었을 수 있다.
- 추가 확인: Train에서 유사한 조건의 주문 건수와 금액 분포를 확인한다.

두 사례에서는 예측값이 실제 고액 주문과 소액 주문의 차이를 충분히 반영하지 못했다.

## 8. Validation Evidence

| check | value | status |
| --- | ---: | --- |
| `forbidden_feature_overlap` | 0 | PASS |
| `strict_train_before_test` | True | PASS |
| `selected_model_exists_in_train_cv` | True | PASS |
| `final_test_contains_baseline` | True | PASS |
| `final_test_contains_frozen_selected_model` | True | PASS |
| `test_rows_for_r2` | 56 | PASS |

### FAIL 항목

- 없음

## 9. Evidence와 공개 범위

### 생성된 Evidence

- [x] `ch09_regression_split_summary.csv`
- [x] `ch09_regression_feature_audit.csv`
- [x] `ch09_regression_cv_summary.csv`
- [x] `ch09_regression_model_comparison.csv`
- [x] `ch09_regression_validation.csv`
- [x] `ch09_regression_checklist.csv`

### Internal 파일

- [x] `ch09_regression_model_data_internal.csv`
- [x] `ch09_regression_predictions_internal.csv`

### 공개 결과

- [x] `ch09_regression_report.md`
- [x] `ch09_actual_vs_predicted.png`
- [x] `ch09_residual_histogram.png`

### 공개 범위 판단

`order_id`가 있으면 예측 결과를 특정 주문 기록과 연결할 수 있다. 식별자가 포함된 원본 결과는 내부 진단용으로 보관하고, 공개 답안에는 식별자를 뺀 오차 수치와 집계 결과를 사용한다.

## 10. 최종 사용 판단

- [ ] 현재 수준에서 참고용 사용 가능
- [ ] 추가 검증 후 사용 가능
- [x] 현재 데이터로는 사용 보류

### 판단 근거

1. Random Forest는 Train CV의 MAE 표준편차가 가장 작았지만, 평균 MAE는 약 437,353원으로 Baseline의 423,005원보다 높았다.
2. Final Test에서도 MAE가 Baseline보다 3.47% 높았고, RMSE는 약 548,190원, R²는 -0.1033이었다.
3. 금지 Feature 중복은 0개였고 시간순 분할도 통과했다. 이 조건에서 고액 주문을 낮게 예측하고 소액 주문을 높게 예측한 사례가 있었다.

### 업무적 위험

예측 금액을 그대로 사용하면 고액 주문의 규모를 작게 보거나 소액 주문의 규모를 크게 판단할 수 있다.

### 현재 데이터의 한계

전체 주문은 300건이고 Final Test는 56건이다. 현재 사용한 주문 날짜·결제수단·고객 특성만으로는 주문별 금액 차이를 충분히 설명하기 어려웠다.

### 다음 개선 우선순위

1. 실제 주문 금액대별로 과대·과소 예측이 얼마나 발생하는지 확인한다.
2. 예측 시점에 확보할 수 있는 과거 구매 정보가 있는지 확인한다. 사용한다면 해당 주문 이전 기록만으로 계산한다.
3. 이후 기간 데이터를 확보해 같은 시간순 검증 절차로 Baseline과 다시 비교한다.

### LLM 활용 및 검증

LLM은 예측 시점 문구와 누수 제외 이유, 결과 해석을 정리하는 데 활용했다. Feature는 실제 Audit 표와 금지 변수 중복 0건을 확인해 검토했다. 모델 선택과 성능 판단은 직접 실행한 Train CV와 Final Test 결과를 기준으로 했다.

## 11. 전체 재실행 확인

```powershell
& ..\llm-data-analysis-course\.venv\Scripts\python.exe scripts/prepare_ch09_data.py
& ..\llm-data-analysis-course\.venv\Scripts\python.exe scripts/run_regression_analysis.py
```

- 실행 성공 여부: 두 스크립트 모두 성공했고, 데이터 준비 시 FK 오류는 0건이었다.
- Train CV Selected Model: Random Forest
- Final Validation 전체 PASS 여부: 6개 모두 PASS
- Notebook 결과와 스크립트 결과의 일관성: Train 244건과 Final Test 56건, 선택 모델, Final Test 지표, Validation 결과가 일치했다.

## 최종 체크

- [x] Target과 예측 시점을 정의했습니다.
- [x] Leakage Audit을 수행했습니다.
- [x] Target 계산과 관계 검증을 확인했습니다.
- [x] Train과 Final Test가 시간 순서로 엄격히 분리되었습니다.
- [x] 전처리가 Pipeline 내부에서 학습됩니다.
- [x] Train TimeSeriesSplit으로 후보를 비교했습니다.
- [x] Final Test 전에 Selected Model을 고정했습니다.
- [x] Final Test에서는 Baseline과 Frozen Model만 비교했습니다.
- [x] MAE, RMSE, R²를 함께 해석했습니다.
- [x] 대표 오차 사례를 관찰과 가설로 구분했습니다.
- [x] Validation Evidence가 모두 PASS입니다.
- [x] Internal 결과와 공개 결과를 구분했습니다.
- [x] 낮은 성능도 숨기지 않았습니다.
- [x] 전체 스크립트를 재실행했습니다.
- [x] 최종 Notebook URL을 제출합니다.